In [1]:
# !pip install nltk

In [2]:
# %pip install spacy

In [3]:
# # import sys
# !{sys.executable} -m spacy download nl_core_news_lg

In [4]:
import pandas as pd
import nltk

In [5]:
df = pd.read_csv("./dataset/cleaned_Leesmateriaal.csv")
SamenKolom = df["Gecombineerd"]

In [6]:
SamenKolom = SamenKolom.str.lower()

In [ ]:
#cijfers met spatie ervoor en erna verwijderen
SamenKolom = SamenKolom.str.replace(r'(?<=\s)\d+(?=\s)', '', regex=True)

In [ ]:
#puntkomma's en schuine strepen vervangen door spaties
SamenKolom = SamenKolom.str.replace(';', ' ')
SamenKolom = SamenKolom.str.replace('/', ' ')

In [ ]:
#alle overige leestekens verwijderen
SamenKolom = SamenKolom.str.replace(r'[^\w\s]', '', regex=True)

In [ ]:
#dubbele spaties verwijderen
SamenKolom = SamenKolom.str.replace(r'\s+', ' ', regex=True)
SamenKolom = SamenKolom.str.strip()

In [11]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('dutch'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\merij\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#functie om stopwoorden te verwijderen
def verwijder_stopwoorden(zin):
    woorden = zin.split()
    nieuwe_woorden = []

    for woord in woorden:
        if woord.lower() not in stop_words:
            nieuwe_woorden.append(woord)

    return ' '.join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(verwijder_stopwoorden)

In [ ]:
#functie om woorden te stammen (SnowballStemmer voor Nederlands)
# from nltk.stem.snowball import SnowballStemmer

# stemmer = SnowballStemmer("dutch")

# def stam_tekst(zin):
#     woorden = zin.split()
#     gestamde_woorden = []

#     for woord in woorden:
#         gestamde_woorden.append(stemmer.stem(woord))

#     return " ".join(gestamde_woorden)

# SamenKolom = SamenKolom.apply(stam_tekst)

In [ ]:
#functie om woorden te lemmatiseren (Spacy voor Nederlands)
import spacy

nlp = spacy.load("nl_core_news_lg")

def lemmatiseer(zin):
    woorden = nlp(zin)
    nieuwe_woorden = []

    for woord in woorden:
        if not woord.is_punct:
            nieuwe_woorden.append(woord.lemma_)

    return " ".join(nieuwe_woorden)

SamenKolom = SamenKolom.apply(lemmatiseer)

In [64]:
df["nlp"] = SamenKolom

In [65]:
df.to_csv("./dataset/cleaned_Leesmateriaal.csv", index=False)

In [20]:
#TF-IDF vectorizer importeren
from sklearn.feature_extraction.text import TfidfVectorizer

teksten = SamenKolom.fillna("").astype(str)

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(teksten)

print(X_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4436 stored elements and shape (283, 1082)>
  Coords	Values
  (0, 172)	0.2088331861016744
  (0, 425)	0.2088331861016744
  (0, 548)	0.1524003777216056
  (0, 873)	0.1602246107505877
  (0, 892)	0.14320092234151008
  (0, 1022)	0.1703118015291988
  (0, 632)	0.3892321784094844
  (0, 39)	0.2088331861016744
  (0, 698)	0.11757332005979104
  (0, 619)	0.1946160892047422
  (0, 310)	0.17670466539714896
  (0, 381)	0.1946160892047422
  (0, 72)	0.2088331861016744
  (0, 44)	0.1946160892047422
  (0, 363)	0.1703118015291988
  (0, 325)	0.1602246107505877
  (0, 173)	0.1703118015291988
  (0, 485)	0.11757332005979104
  (0, 959)	0.2088331861016744
  (0, 269)	0.04879036315148205
  (0, 1041)	0.4176663722033488
  (0, 2)	0.06527041779804334
  (0, 229)	0.085186025669289
  (0, 542)	0.08367225065293186
  (0, 692)	0.09262564414391751
  :	:
  (281, 495)	0.19966053760083052
  (281, 708)	0.20450049474583976
  (281, 636)	0.6481354566096464
  (281, 234)	0.20991

In [19]:
#TF-IDF resultaten omzetten naar DataFrame
import pandas as pd

tfidf_df = pd.DataFrame.sparse.from_spmatrix(
    X_tfidf,
    index=SamenKolom.index,
    columns=vectorizer.get_feature_names_out()
)

tfidf_df.head()

,2f,2f3f,3f,aandacht,aangrijpen,aankijken,aanleiding,aantrekkingkracht,aanwijzing,abba,...,zomers,zorg,zorgen,zoulfa,zurhake,zutphen,zwaar,zwangerschap,én,öztürk
0,NaN,NaN,0.06527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,0.074411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.082909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,0.07213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.081024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
SamenKolom[0:3]

0    dagboek joods meisje tijdens twee Wereldoorlog...
1    spannend thriller waarin vrouw confronteren st...
2    toegankelijk verhaal liefd relatie zoektocht p...
Name: Gecombineerd, dtype: str

In [21]:
intdf = pd.read_csv("./dataset/cleaned_Interacties.csv")

In [43]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", intdf.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": intdf.nunique(),
    "Aantal missende waarden": intdf.isna().sum(),
    "Datatype": intdf.dtypes,
    "Aantal dubbele waarden": intdf.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (2000, 8)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
interactie_id,2000,0,str,0
student_id,100,0,str,1900
cataloguscode,276,0,str,1724
interactietype,6,0,str,1994
beoordeling,5,373,float64,1622
leespercentage,101,0,int64,1899
leestijd_min,560,0,int64,1440
datum,288,0,str,1712


In [58]:
intdf[1995:2000]

,interactie_id,student_id,cataloguscode,interactietype,beoordeling,leespercentage,leestijd_min,datum
1995,I01996,S100,CAT-00030,gelezen,4.0,99,261,2026-04-30
1996,I01997,S100,CAT-00250,niet_afgemaakt,3.0,60,98,2026-07-16
1997,I01998,S100,CAT-00066,gelezen,4.0,96,698,2026-09-03
1998,I01999,S100,CAT-00009,bekeken,NaN,57,318,2026-01-08
1999,I02000,S100,CAT-00143,bekeken,3.0,37,3,2026-10-15


In [ ]:
intdf["interactietype"].unique().tolist()

['opgeslagen', 'bekeken', 'gelezen', 'niet_afgemaakt', 'geliket', 'afgewezen']

In [37]:
inttype = intdf["interactietype"]

In [40]:
inttype.unique().tolist()

[4.0, 2.0, 3.0, 1.0, 5.0]

In [ ]:
#interactietypes waardes geven van 1 tot 5
inttype.loc[inttype == "geliket"] = "5"
inttype.loc[inttype == "opgeslagen"] = "4"
inttype.loc[inttype == "gelezen"] = "3"
inttype.loc[inttype == "bekeken"] = "2"
inttype.loc[inttype == "niet_afgemaakt"] = "1"
inttype.loc[inttype == "afgewezen"] = "1"
inttype = inttype.astype(float)

In [53]:
leesprc = intdf["leespercentage"]

In [ ]:
#leespercentage waardes geven van 1 tot 5
leesprc = leesprc.apply(lambda x: round(x / 20))
leesprc = leesprc.astype(float)

In [55]:
leesprc.unique().tolist()

[4.0, 2.0, 1.0, 5.0, 0.0, 3.0]

In [69]:
#interacties score berekenen, lege waardes worden niet meegenomen
scores = pd.DataFrame({
    "leesprc": leesprc,
    "inttype": inttype,
    "beoordeling": intdf["beoordeling"],
}, index=intdf.index)

weights = pd.Series({
    "leesprc": 0.15,
    "inttype": 0.35,
    "beoordeling": 0.50,
})

weighted_sum = scores.mul(weights, axis="columns").sum(axis=1, min_count=1)
available_weight = scores.notna().mul(weights, axis="columns").sum(axis=1)

intscore = weighted_sum / available_weight

In [70]:
intscore

0       4.00
1       2.50
2       4.00
3       3.65
4       1.50
        ... 
1995    3.80
1996    2.30
1997    3.80
1998    2.30
1999    2.50
Length: 2000, dtype: float64

In [71]:
intdf["intscore"] = intscore

In [62]:
intdf.head()

,interactie_id,student_id,cataloguscode,interactietype,beoordeling,leespercentage,leestijd_min,datum,intscore
0,I00001,S001,CAT-00223,opgeslagen,4.0,88,305,2026-01-17,4.00
1,I00002,S001,CAT-00253,bekeken,3.0,42,93,2026-08-24,2.50
2,I00003,S001,CAT-00055,opgeslagen,4.0,84,299,2026-10-14,4.00
3,I00004,S001,CAT-00266,gelezen,4.0,82,235,2026-05-04,3.75
4,I00005,S001,CAT-00262,niet_afgemaakt,2.0,14,2,2026-02-19,1.50


In [72]:
intdf.to_csv("./dataset/cleaned_Interacties.csv", index=False)

In [73]:
intdf["intscore"] = intscore

student_item_matrix = intdf.pivot_table(
    index="student_id",
    columns="cataloguscode",
    values="intscore",
    aggfunc="mean"
)

student_item_matrix.head()

cataloguscode,CAT-00001,CAT-00002,CAT-00003,CAT-00004,CAT-00005,CAT-00006,CAT-00007,CAT-00008,CAT-00009,CAT-00010,...,CAT-00274,CAT-00275,CAT-00276,CAT-00277,CAT-00278,CAT-00279,CAT-00280,CAT-00281,CAT-00282,CAT-00283
student_id,,,,,,,,,,,,,,,,,,,,,
S001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.65,NaN,NaN
S002,NaN,NaN,NaN,NaN,4.15,NaN,NaN,NaN,2.15,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
S003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
S004,1.65,NaN,NaN,NaN,NaN,NaN,2.8,NaN,2.50,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN
S005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
student_item_matrix.to_csv("./matrix/student_item_matrix.csv", index=True)

In [ ]:
# Vergelijk de scorepatronen van alle studenten op materialen.
# .T draait de matrix om, zodat corr() de studenten met elkaar vergelijkt.
# min_periods=3 vereist minstens 3 materialen met scores van beide studenten.
student_similarity = student_item_matrix.T.corr(min_periods=3)

# Maak een matrix waarin 1 betekent dat een score bekend is en 0 dat die ontbreekt.
# De matrixvermenigvuldiging telt per studentenpaar hoeveel materialen zij delen.
overlap = (
    student_item_matrix.notna().astype(int)
    @ student_item_matrix.notna().astype(int).T
)

# Kies voor dit voorbeeld de eerste student uit de matrix.
# De volgende cel gebruikt expliciet S001.
student_id = student_similarity.index[0]
# Pak de correlaties van deze student met alle andere studenten.
neighbors = (
    student_similarity[student_id]
    # Verwijder de student zelf: die is altijd volledig gelijk aan zichzelf.
    .drop(index=student_id)
    # Verwijder studenten waarvoor geen correlatie berekend kon worden.
    .dropna()
    # Zet de hoogste correlaties bovenaan.
    .sort_values(ascending=False)
    # Bewaar de 5 hoogste correlaties (die kunnen ook negatief zijn).
    .head(5)
)
# Toon de meest vergelijkbare studenten en het aantal gedeelde materialen
# Toon de gekozen student en de correlaties met diens vergelijkbare studenten.
print("Student:", student_id)
print("Vergelijkbare studenten:")
print(neighbors)
# Toon hoeveel materialen deze studenten met de gekozen student gemeen hebben.
print("Gedeelde materialen:")
print(overlap.loc[student_id, neighbors.index])

Student: S001
Vergelijkbare studenten:
student_id
S051    0.964056
S038    0.911373
S002    0.900643
S006    0.805943
S014    0.790474
Name: S001, dtype: float64
Gedeelde materialen:
student_id
S051    3
S038    4
S002    3
S006    3
S014    3
Name: S001, dtype: int32


In [81]:
# Kies de student voor wie we aanbevelingen willen maken.
target = "S001"

# Haal de correlaties van deze student met anderen op.
# Verwijder de student zelf en correlaties die niet berekend konden worden.
neighbors = student_similarity[target].drop(target).dropna()
# Houd alleen positief vergelijkbare studenten over en kies de 5 sterkste buren.
neighbors = neighbors[neighbors > 0].nlargest(5)

# Haal de scores op die deze buren aan alle materialen hebben gegeven.
neighbor_scores = student_item_matrix.loc[neighbors.index]

# Vermenigvuldig elke score met de gelijkenis van de buur en tel dit per materiaal op.
# min_count=1 zorgt dat een materiaal zonder scores NaN blijft in plaats van 0.
weighted_sum = neighbor_scores.mul(neighbors, axis=0).sum(axis=0, min_count=1)
# Tel per materiaal de gelijkenissen op van de buren die het hebben beoordeeld.
weight_sum = neighbor_scores.notna().mul(neighbors, axis=0).sum(axis=0)
# Deel de gewogen scoresom door de gebruikte gewichten voor een voorspelde score.
predicted_scores = weighted_sum / weight_sum

# Tel hoeveel buren voor elk materiaal een score hebben gegeven.
support = neighbor_scores.notna().sum(axis=0)
# Bepaal welke materialen de gekozen student nog niet heeft beoordeeld.
unseen = student_item_matrix.loc[target].isna()

# Houd alleen onbekende materialen over die door minstens 2 buren zijn beoordeeld.
# Sorteer op voorspelde score en selecteer de 10 hoogste aanbevelingen.
recommendations = (
    predicted_scores[unseen & (support >= 2)]
    .sort_values(ascending=False)
    .head(10)
)

# Toon de aanbevolen cataloguscodes met hun voorspelde scores.
print(recommendations)

cataloguscode
CAT-00156    4.318459
CAT-00181    3.830951
CAT-00128    3.727551
CAT-00264    3.650000
CAT-00034    3.269304
CAT-00030    3.058847
CAT-00195    3.041780
CAT-00079    3.028745
CAT-00202    2.851141
CAT-00157    2.806359
dtype: float64
